# Required Packages

In [ ]:
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from pathlib import Path

# Project Path

In [ ]:
PROJECT_DIR = Path.cwd().parent
print(PROJECT_DIR)

# Helper functions

In [ ]:
# Helper functions for color composite by Martin Raspaud
def read_band(path):
    with rasterio.open(path) as src:
        arr = src.read(1).astype("float32")
        profile = src.profile.copy()
        nodata = src.nodata
    return arr, profile, nodata

def db_to_linear(x_db):
    return 10 ** (x_db / 10.0)

def stretch(arr, vmin, vmax):
    out = (arr - vmin) / (vmax - vmin)
    return np.clip(out, 0, 1)

def gamma_corr(arr, gamma=1.1):
    arr = np.clip(arr, 0, 1)
    return arr ** (1.0 / gamma)

def overlay(top, bottom):
    return ((1 - 2 * top) * bottom + 2 * top) * bottom

def sar_ice_rgb(hh, hv, input_in_db=False):
    """
    hh, hv: 2D numpy arrays
    input_in_db=False, wif data is already linear
    """
    hh = hh.astype("float32")
    hv = hv.astype("float32")

    if input_in_db:
        hh = db_to_linear(hh)
        hv = db_to_linear(hv)

    # mask invalid values
    mask = ~np.isfinite(hh) | ~np.isfinite(hv)

    mhv = np.sqrt(hv + 0.002)
    mhh = np.sqrt(hh + 0.002)
    ov = overlay(mhh, mhv)

    red   = gamma_corr(stretch(mhv, 0.02, 0.10), 1.1)
    green = gamma_corr(stretch(ov,  0.00, 0.06), 1.1)
    blue  = gamma_corr(stretch(mhh, 0.00, 0.32), 1.1)

    rgb = np.dstack([red, green, blue])

    rgb[mask] = 0
    return rgb

# Usage

## 02DC20

In [ ]:
hh_path = PROJECT_DIR / "output" / "final_tifs" / "HH_02DC20_final.tif"
hv_path = PROJECT_DIR / "output" / "final_tifs" / "HV_02DC20_final.tif"

hh, profile, hh_nodata = read_band(hh_path)
hv, _, hv_nodata = read_band(hv_path)

print(hh.shape, hv.shape)
assert hh.shape == hv.shape, "HH and HV have different dimensions."

rgb = sar_ice_rgb(hh, hv, input_in_db=False)

plt.figure(figsize=(10, 10))
plt.imshow(rgb)
plt.title("Sentinel-1 SAR Ice Composite (HH/HV)")
plt.axis("off")
plt.show()



In [ ]:
#Export
out_path = PROJECT_DIR / "output" / "final_tifs" / "02DC20_RGB.tif"

out_profile = profile.copy()
out_profile.update(
    dtype="uint8",
    count=3,
    compress="lzw",
    driver="GTiff"
)

with rasterio.open(out_path, "w", **out_profile) as dst:
    for i in range(3):
        band = np.clip(rgb[:, :, i], 0, 1)
        band = (band * 255).astype("uint8")
        dst.write(band, i + 1)

print("Saved:", out_path)

## 02DC57

In [ ]:
hh_path = PROJECT_DIR / "output" / "final_tifs" / "HH_02DC57_final.tif"
hv_path = PROJECT_DIR / "output" / "final_tifs" / "HV_02DC57_final.tif"

hh, profile, hh_nodata = read_band(hh_path)
hv, _, hv_nodata = read_band(hv_path)

print(hh.shape, hv.shape)
assert hh.shape == hv.shape, "HH and HV have different dimensions."

rgb = sar_ice_rgb(hh, hv, input_in_db=False)

plt.figure(figsize=(10, 10))
plt.imshow(rgb)
plt.title("Sentinel-1 SAR Ice Composite (HH/HV)")
plt.axis("off")
plt.show()

In [ ]:
out_path = PROJECT_DIR / "output" / "final_tifs" / "02DC57_RGB.tif"

out_profile = profile.copy()
out_profile.update(
    dtype="uint8",
    count=3,
    compress="lzw",
    driver="GTiff"
)

with rasterio.open(out_path, "w", **out_profile) as dst:
    for i in range(3):
        band = np.clip(rgb[:, :, i], 0, 1)
        band = (band * 255).astype("uint8")
        dst.write(band, i + 1)

print("Saved:", out_path)

# Sources

[Raspaud, M. (2020): SAR-Ice: A Sea Ice RGB Composite ](https://custom-scripts.sentinel-hub.com/custom-scripts/sentinel-1/sar-ice/)


Supplementary Material: [Raspaud, M. & Itkin, M.(2020): SAR-Ice: A Sea Ice RGB Composite ](https://custom-scripts.sentinel-hub.com/custom-scripts/sentinel-1/sar-ice/supplementary_material.pdf)